# 01 · Validação dos dados

Auditoria da base bruta **sem alterar nenhum dado**. O objetivo é levantar todos os problemas de qualidade e decidir o que será corrigido na etapa 2 (limpeza).

> **Aviso:** esta base é **sintética**. Ela foi gerada por uma simulação, em um ambiente controlado, e não coletada de pessoas reais. Por isso, ela não traz os problemas típicos de dados do mundo real, como erros de digitação, campos em branco ou registros duplicados, e há pouco a corrigir. A validação foi feita mesmo assim, por completo: verificar a qualidade dos dados antes de qualquer análise é um bom hábito, e ele deve ser mantido independentemente da origem da base.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from scroll_sono.config import (BINARY_COLS, CATEGORICAL_COLS, EXPECTED_DOMAINS, ID_COL,
                                MOMENT, NUMERIC_COLS, ORDINAL_COLS, TARGET, TARGET_CLASSES)
from scroll_sono.data import load_raw

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
df = load_raw()
print(f"{df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head()

8,500 linhas × 18 colunas


,user_id,age,gender,occupation_type,chronotype,bedtime_phone_minutes,primary_bedtime_app,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score,sleep_debt_category
0,USR-00001,29,Female,Healthcare / Shift Worker,Intermediate,179,Instagram / Reddit,54,1,0,21,84.4,3.20,23.5,22.2,7,10.0,Severe Sleep Debt
1,USR-00002,58,Female,Student,Intermediate,163,YouTube,76,1,92,72,86.7,4.06,18.0,21.5,7,10.0,Severe Sleep Debt
2,USR-00003,41,Female,Healthcare / Shift Worker,Night Owl,100,TikTok / Reels,59,0,45,23,70.5,3.20,18.1,20.5,7,10.0,Severe Sleep Debt
3,USR-00004,36,Non-Binary,Remote Tech,Intermediate,34,YouTube,52,0,97,49,38.1,6.99,21.6,22.4,1,1.7,Mild Deficit
4,USR-00005,23,Female,Healthcare / Shift Worker,Intermediate,27,TikTok / Reels,82,1,0,38,32.3,4.58,24.1,22.8,4,5.3,Moderate Debt


## 1. Visão geral e tipos

O tipo lido pelo pandas é comparado com o tipo esperado segundo o dicionário de dados do README.

In [3]:
esperado = {ID_COL: "identificador", TARGET: "alvo (ordinal)"}
esperado |= {c: "numérico" for c in NUMERIC_COLS}
esperado |= {c: "categórico nominal" for c in CATEGORICAL_COLS if c not in ORDINAL_COLS}
esperado |= {c: "categórico ordinal" for c in ORDINAL_COLS}
esperado |= {c: "binário" for c in BINARY_COLS}

tipos = pd.DataFrame({
    "dtype lido": df.dtypes.astype(str),
    "tipo esperado": pd.Series(esperado),
    "exemplo": df.iloc[0],
})
tipos

,dtype lido,tipo esperado,exemplo
age,int64,numérico,29
bedtime_phone_minutes,int64,numérico,179
blue_light_filter_active,int64,binário,1
caffeine_post_5pm_mg,int64,numérico,0
chronotype,str,categórico ordinal,Intermediate
deep_sleep_pct,float64,numérico,23.5
gender,str,categórico nominal,Female
morning_alarm_snoozes,int64,numérico,7
next_day_fatigue_score,float64,numérico,10.0
occupation_type,str,categórico nominal,Healthcare / Shift Worker


In [4]:
# Toda coluna numérica esperada foi lida como número?
nao_numericas = [c for c in NUMERIC_COLS + BINARY_COLS if not pd.api.types.is_numeric_dtype(df[c])]
print("Colunas numéricas lidas como texto:", nao_numericas or "nenhuma")

Colunas numéricas lidas como texto: nenhuma


**Resultado:** todas as colunas foram lidas com o tipo esperado. As numéricas inteiras vieram como `int64`, as contínuas como `float64` e as categóricas como texto.

## 2. Valores ausentes

In [5]:
ausentes = pd.DataFrame({
    "ausentes": df.isna().sum(),
    "% ausentes": (df.isna().mean() * 100).round(2),
})
ausentes

,ausentes,% ausentes
user_id,0,0.0
age,0,0.0
gender,0,0.0
occupation_type,0,0.0
chronotype,0,0.0
bedtime_phone_minutes,0,0.0
primary_bedtime_app,0,0.0
screen_brightness_pct,0,0.0
blue_light_filter_active,0,0.0
caffeine_post_5pm_mg,0,0.0


In [6]:
# Valores que representam ausência disfarçada de texto
marcadores = ["", "na", "n/a", "nan", "null", "none", "-", "?", "unknown"]
texto = df.select_dtypes(include=["str", "object"])
disfarcados = texto.apply(lambda s: s.str.strip().str.lower().isin(marcadores).sum())
print("Ausentes disfarçados por coluna categórica:")
disfarcados

Ausentes disfarçados por coluna categórica:


user_id                0
gender                 0
occupation_type        0
chronotype             0
primary_bedtime_app    0
sleep_debt_category    0
dtype: int64

**Resultado:** nenhum valor ausente, nem como `NaN` nem disfarçado de texto (`"N/A"`, `"?"`, `""` etc.).

## 3. Duplicatas

In [7]:
print("Linhas totalmente duplicadas:", df.duplicated().sum())
print(f"IDs repetidos ({ID_COL}):", df[ID_COL].duplicated().sum())
print("Linhas iguais ignorando o ID:", df.drop(columns=ID_COL).duplicated().sum())
print("Todo ID segue o formato USR-00000:", df[ID_COL].str.fullmatch(r"USR-\d{5}").all())

Linhas totalmente duplicadas: 0
IDs repetidos (user_id): 0
Linhas iguais ignorando o ID: 0
Todo ID segue o formato USR-00000: True


**Resultado:** nenhuma duplicata. Cada `user_id` é único e segue o mesmo formato. A checagem ignorando o ID é importante porque o mesmo participante poderia ter sido registrado duas vezes com IDs diferentes.

## 4. Valores mal formatados

Para cada coluna categórica, verifica se há espaços extras e se existem categorias que só diferem por maiúsculas/minúsculas ou espaços (ex.: `"SP"`, `"SP "`, `"sp"`).

In [8]:
linhas = []
for col in CATEGORICAL_COLS + [TARGET]:
    s = df[col]
    normalizado = s.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
    linhas.append({
        "coluna": col,
        "categorias": s.nunique(),
        "categorias após normalizar": normalizado.nunique(),
        "valores com espaços extras": (s != s.str.strip()).sum(),
    })
pd.DataFrame(linhas).set_index("coluna")

,categorias,categorias após normalizar,valores com espaços extras
coluna,,,
gender,3,3,0
occupation_type,5,5,0
primary_bedtime_app,6,6,0
chronotype,3,3,0
sleep_debt_category,4,4,0


In [9]:
for col in CATEGORICAL_COLS + [TARGET]:
    print(f"{col}: {sorted(df[col].unique())}")

gender: ['Female', 'Male', 'Non-Binary']
occupation_type: ['Corporate 9-to-5', 'Freelance / Creative', 'Healthcare / Shift Worker', 'Remote Tech', 'Student']
primary_bedtime_app: ['Instagram / Reddit', 'Messaging / Chat', 'News / Reading', 'Streaming (Netflix/Hulu)', 'TikTok / Reels', 'YouTube']
chronotype: ['Intermediate', 'Morning Lark', 'Night Owl']
sleep_debt_category: ['Mild Deficit', 'Moderate Debt', 'Optimal Recovery', 'Severe Sleep Debt']


In [10]:
# As categorias ordinais e do alvo batem exatamente com a documentação?
for col, ordem in ORDINAL_COLS.items():
    print(col, "→ fora do esperado:", set(df[col]) - set(ordem) or "nenhuma")
print(TARGET, "→ fora do esperado:", set(df[TARGET]) - set(TARGET_CLASSES) or "nenhuma")

chronotype → fora do esperado: nenhuma
sleep_debt_category → fora do esperado: nenhuma


**Resultado:** nenhuma inconsistência de formatação. Cada categoria aparece escrita de uma única forma, e as categorias de `chronotype` e do alvo são exatamente as documentadas.

## 5. Valores fora do domínio

O domínio esperado vem da documentação (ex.: brilho entre 10% e 100%, cansaço entre 1 e 10) ou, quando ela não define, de limites físicos plausíveis (ex.: no máximo 24 horas de sono, porcentagens entre 0 e 100). Os limites estão em `scroll_sono.config.EXPECTED_DOMAINS`.

In [11]:
linhas = []
for col, (minimo, maximo, inteiro) in EXPECTED_DOMAINS.items():
    s = df[col]
    abaixo = (s < minimo).sum()
    acima = (s > maximo).sum() if maximo is not None else 0
    nao_inteiros = (s != s.round()).sum() if inteiro else 0
    linhas.append({
        "coluna": col,
        "domínio esperado": f"{minimo} – {maximo if maximo is not None else '∞'}",
        "mín. observado": s.min(),
        "máx. observado": s.max(),
        "abaixo": abaixo,
        "acima": acima,
        "não inteiros": nao_inteiros,
        "ok": abaixo + acima + nao_inteiros == 0,
    })
dominio = pd.DataFrame(linhas).set_index("coluna")
dominio

,domínio esperado,mín. observado,máx. observado,abaixo,acima,não inteiros,ok
coluna,,,,,,,
age,0 – 120,18.0,65.0,0,0,0,True
bedtime_phone_minutes,0 – 1440,1.0,180.0,0,0,0,True
screen_brightness_pct,10 – 100,10.0,100.0,0,0,0,True
blue_light_filter_active,0 – 1,0.0,1.0,0,0,0,True
caffeine_post_5pm_mg,0 – 1000,0.0,250.0,0,0,0,True
physical_activity_min,0 – 1440,0.0,112.0,0,0,0,True
sleep_latency_min,0 – 1440,6.0,123.3,0,0,0,True
total_sleep_hours,0 – 24,3.2,9.8,0,0,0,True
deep_sleep_pct,0 – 100,8.1,28.0,0,0,0,True


In [12]:
# A coluna binária só tem 0 e 1?
for col in BINARY_COLS:
    print(col, sorted(df[col].unique()))

blue_light_filter_active [np.int64(0), np.int64(1)]


**Resultado:** todos os valores estão dentro do domínio esperado. Não há horas negativas, porcentagens acima de 100 nem contagens fracionárias.

## 6. Consistência entre colunas

Regras que relacionam colunas diferentes e que não podem ser violadas:

- **A:** sono profundo + REM não pode passar de 100% do sono total.
- **B:** tempo para adormecer + horas dormidas não pode passar de 24h.
- **C:** uso do celular na cama + tempo para adormecer + horas dormidas não pode passar de 24h.

In [13]:
horas_latencia = df["sleep_latency_min"] / 60
horas_celular = df["bedtime_phone_minutes"] / 60

regras = {
    "A: deep + REM ≤ 100%": df["deep_sleep_pct"] + df["rem_sleep_pct"] <= 100,
    "B: latência + sono ≤ 24h": horas_latencia + df["total_sleep_hours"] <= 24,
    "C: celular + latência + sono ≤ 24h": horas_celular + horas_latencia + df["total_sleep_hours"] <= 24,
}
pd.DataFrame({"violações": {nome: (~ok).sum() for nome, ok in regras.items()}})

,violações
A: deep + REM ≤ 100%,0
B: latência + sono ≤ 24h,0
C: celular + latência + sono ≤ 24h,0


In [14]:
(df["deep_sleep_pct"] + df["rem_sleep_pct"]).describe().round(2)

count    8500.00
mean       40.84
std         4.30
min        23.40
25%        38.10
50%        41.10
75%        43.80
max        53.80
dtype: float64

**Resultado:** nenhuma violação. A soma de sono profundo + REM fica entre 23% e 54%, o que é coerente: o restante do sono corresponde às fases leves.

## 7. Cardinalidade e colunas constantes

Colunas com um único valor (ou quase isso) não ajudam o modelo. Colunas categóricas com muitas categorias geram muitas colunas no one-hot.

In [15]:
card = pd.DataFrame({
    "valores únicos": df.nunique(),
    "valor mais frequente": df.mode().iloc[0],
    "% do mais frequente": (df.apply(lambda s: s.value_counts(normalize=True).iloc[0]) * 100).round(1),
})
card.sort_values("% do mais frequente", ascending=False)

,valores únicos,valor mais frequente,% do mais frequente
caffeine_post_5pm_mg,218,0.0,58.2
blue_light_filter_active,2,0.0,53.2
sleep_debt_category,4,Moderate Debt,52.5
gender,3,Female,51.1
chronotype,3,Intermediate,45.6
occupation_type,5,Corporate 9-to-5,33.4
primary_bedtime_app,6,TikTok / Reels,25.9
next_day_fatigue_score,91,1.0,24.8
morning_alarm_snoozes,8,2.0,22.0
physical_activity_min,107,0.0,5.5


**Resultado:**

- Nenhuma coluna é constante.
- `user_id` tem um valor diferente por linha (8.500): é um identificador e não tem valor preditivo.
- As categóricas têm baixa cardinalidade (de 3 a 6 categorias), então o one-hot gera poucas colunas.
- `caffeine_post_5pm_mg` tem **58% de zeros** (pessoas que não consomem cafeína à noite). Não é erro, mas é uma distribuição bem assimétrica, a ser analisada na EDA.

## 8. Valores acumulados nos limites

Como o dataset é **sintético**, a simulação pode ter "cortado" os valores no mínimo ou no máximo permitidos (*clipping*). Isso aparece como um acúmulo de registros exatamente no limite. Não são erros, mas distorcem a distribuição e são relevantes para a EDA.

In [16]:
linhas = []
for col in NUMERIC_COLS:
    s = df[col]
    linhas.append({
        "coluna": col,
        "mínimo": s.min(),
        "% no mínimo": round((s == s.min()).mean() * 100, 1),
        "máximo": s.max(),
        "% no máximo": round((s == s.max()).mean() * 100, 1),
    })
limites = pd.DataFrame(linhas).set_index("coluna")
limites.sort_values(["% no mínimo", "% no máximo"], ascending=False)

,mínimo,% no mínimo,máximo,% no máximo
coluna,,,,
caffeine_post_5pm_mg,0.0,58.2,250.0,0.6
next_day_fatigue_score,1.0,24.8,10.0,3.7
morning_alarm_snoozes,0.0,8.9,7.0,2.1
physical_activity_min,0.0,5.5,112.0,0.0
total_sleep_hours,3.2,2.8,9.8,0.2
age,18.0,2.7,65.0,0.5
screen_brightness_pct,10.0,1.4,100.0,1.5
deep_sleep_pct,8.1,0.0,28.0,1.9
bedtime_phone_minutes,1.0,0.0,180.0,1.0


**Resultado:**

- `caffeine_post_5pm_mg` = 0 em 58,2%, `morning_alarm_snoozes` = 0 em 8,9% e `physical_activity_min` = 0 em 5,5%: zeros naturais (não consumir cafeína, não adiar o despertador, não se exercitar).
- `next_day_fatigue_score` = 1,0 (o mínimo da escala) em **24,8%** dos registros, e = 10,0 (o máximo) em 3,7%. Junto com `total_sleep_hours` = 3,2 em 2,8%, são sinais de *clipping* da simulação.
- As demais colunas têm menos de 3% dos registros em cada limite.

Nenhum desses casos é erro, então nada é corrigido na limpeza. Eles serão considerados na EDA.

## 9. Distribuição da variável alvo

In [17]:
alvo = pd.DataFrame({
    "registros": df[TARGET].value_counts(),
    "%": (df[TARGET].value_counts(normalize=True) * 100).round(1),
}).reindex(TARGET_CLASSES)
alvo

,registros,%
sleep_debt_category,,
Optimal Recovery,1387,16.3
Mild Deficit,2004,23.6
Moderate Debt,4462,52.5
Severe Sleep Debt,647,7.6


In [18]:
razao = alvo["registros"].max() / alvo["registros"].min()
print(f"Razão entre a maior e a menor classe: {razao:.1f}×")

Razão entre a maior e a menor classe: 6.9×


**Resultado:** as classes são **desbalanceadas**. `Moderate Debt` tem 52,5% dos registros, cerca de 6,9 vezes mais que `Severe Sleep Debt` (7,6%). Consequências:

- A separação treino/teste (etapa 3) e os folds da validação cruzada (etapa 6) devem ser **estratificados**.
- A acurácia sozinha engana: um modelo que sempre responde `Moderate Debt` já acerta 52,5%. É preciso usar acurácia balanceada, F1 macro e matriz de confusão.

## 10. Identificadores e vazamento de informação (*data leakage*)

O dicionário de dados classifica cada coluna pelo **momento** em que a informação é obtida. As colunas medidas **durante o sono** ou **no dia seguinte** são consequências da noite de sono, assim como o próprio alvo. Se elas definirem o alvo, o modelo vai "acertar" sem aprender nada sobre os hábitos antes de dormir.

A verificação abaixo testa essa suspeita.

In [19]:
pd.DataFrame(
    [(momento, col) for momento, cols in MOMENT.items() for col in cols],
    columns=["momento", "coluna"],
).groupby("momento", sort=False)["coluna"].apply(", ".join).to_frame()

,coluna
momento,
perfil,"age, gender, occupation_type, chronotype"
antes de dormir,"bedtime_phone_minutes, primary_bedtime_app, sc..."
durante o sono,"sleep_latency_min, total_sleep_hours, deep_sle..."
dia seguinte,"morning_alarm_snoozes, next_day_fatigue_score"


### 10.1 Faixa de cada coluna por classe

Se as faixas de valores de uma coluna **não se sobrepõem** entre as classes, essa coluna sozinha separa o alvo.

In [20]:
faixas = df.groupby(TARGET)["total_sleep_hours"].agg(["min", "max", "mean"]).reindex(TARGET_CLASSES)
faixas.round(2)

,min,max,mean
sleep_debt_category,,,
Optimal Recovery,7.75,9.80,8.35
Mild Deficit,6.75,7.74,7.23
Moderate Debt,3.20,6.74,5.55
Severe Sleep Debt,3.20,5.38,3.76


`total_sleep_hours` separa sozinha três grupos sem nenhuma sobreposição:

- `Optimal Recovery`: 7,75h ou mais
- `Mild Deficit`: de 6,75h a 7,74h
- `Moderate Debt` e `Severe Sleep Debt`: menos de 6,75h

Falta separar `Moderate Debt` de `Severe Sleep Debt`. Dentro desse grupo, vejamos o cansaço no dia seguinte:

In [21]:
pouco_sono = df[df["total_sleep_hours"] < 6.75]
pouco_sono.groupby(TARGET)[["total_sleep_hours", "next_day_fatigue_score"]].agg(["min", "max"]).round(2)

total_sleep_hours       next_day_fatigue_score      
                                  min   max                    min   max
sleep_debt_category                                                     
Moderate Debt                     3.2  6.74                    1.0  10.0
Severe Sleep Debt                 3.2  5.38                    8.6  10.0

In [22]:
pd.crosstab(
    [pouco_sono["total_sleep_hours"] <= 5.38, pouco_sono["next_day_fatigue_score"] >= 8.6],
    pouco_sono[TARGET],
    rownames=["sono ≤ 5,38h", "cansaço ≥ 8,6"],
)

sleep_debt_category         Moderate Debt  Severe Sleep Debt
sono ≤ 5,38h cansaço ≥ 8,6                                  
False        False                   2766                  0
             True                      10                  0
True         False                   1686                  0
             True                       0                647

### 10.2 Uma regra simples reproduz o alvo

Com os limites encontrados acima, uma regra de quatro linhas que usa **só duas colunas**:

In [23]:
def regra(linha):
    if linha["total_sleep_hours"] >= 7.75:
        return "Optimal Recovery"
    if linha["total_sleep_hours"] >= 6.75:
        return "Mild Deficit"
    if linha["total_sleep_hours"] <= 5.38 and linha["next_day_fatigue_score"] >= 8.6:
        return "Severe Sleep Debt"
    return "Moderate Debt"

previsto = df.apply(regra, axis=1)
print(f"Acertos da regra: {(previsto == df[TARGET]).mean():.2%}")

Acertos da regra: 100.00%


### 10.3 Conclusão sobre leakage

O alvo `sleep_debt_category` foi **gerado pela simulação a partir de `total_sleep_hours` e `next_day_fatigue_score`**. Uma regra manual com essas duas colunas reproduz o alvo em 100% dos casos.

Se essas colunas entrarem no modelo, qualquer algoritmo vai chegar perto de 100% de acerto, mas sem responder à pergunta do projeto (*os hábitos antes de dormir afetam o sono?*). As outras colunas medidas durante o sono ou no dia seguinte (`sleep_latency_min`, `deep_sleep_pct`, `rem_sleep_pct`, `morning_alarm_snoozes`) também são consequências da noite, e não algo que se conhece **antes** de dormir.

**Decisão:** os modelos são treinados em dois cenários de atributos, definidos em `scroll_sono.config.FEATURE_SETS`:

- **Comportamental (principal):** apenas as colunas de **perfil** e **antes de dormir** (10 atributos). Está de acordo com a sugestão do autor do dataset de priorizar os atributos comportamentais.
- **Completo (comparação):** todas as 16 colunas, exceto o identificador e o alvo. Serve para medir quanto do desempenho vem do vazamento.

Todas as colunas continuam na base limpa, inclusive para a EDA.

**Uso da base inteira:** esta investigação foi feita antes da separação treino/teste, com todos os registros, porque faz parte da auditoria das colunas. Ela não muda a avaliação do cenário principal: as colunas do cenário comportamental foram escolhidas pelo momento em que cada informação é obtida (perfil e antes de dormir), e não pelos limites encontrados acima, que só confirmaram a suspeita. Ainda assim, um procedimento mais rigoroso faria essa investigação só no conjunto de treino.

**Identificador:** `user_id` não tem valor preditivo e deve ser removido dos preditores.

## Resumo e ações para a etapa 2

| # | Verificação | Resultado | Ação na limpeza |
|---|---|---|---|
| 1 | Tipos | Todos corretos | Nenhuma |
| 2 | Valores ausentes | Nenhum (nem disfarçado) | Nenhuma |
| 3 | Duplicatas | Nenhuma linha ou ID repetido | Nenhuma |
| 4 | Formatação | Categorias consistentes | Nenhuma |
| 5 | Domínio | Todos os valores válidos | Nenhuma |
| 6 | Consistência entre colunas | Nenhuma violação | Nenhuma |
| 7 | Cardinalidade | `user_id` é identificador; categóricas com 3 a 6 categorias | Remover `user_id` dos preditores |
| 8 | Valores nos limites | *Clipping* em `next_day_fatigue_score` e `total_sleep_hours`; 58% de zeros em cafeína | Nenhuma (analisar na EDA) |
| 9 | Distribuição do alvo | Desbalanceado (52,5% × 7,6%) | Estratificar na etapa 3 |
| 10 | Leakage | Alvo gerado a partir de `total_sleep_hours` e `next_day_fatigue_score` | Nenhuma (os cenários de atributos são aplicados na modelagem) |

A base não tem problemas de qualidade: por ser sintética, ela já vem limpa. A descoberta principal desta etapa é o **vazamento de informação**, que define quais colunas podem ser usadas como preditores.